# Audio classification - Finetuning DistilHuBERT model for Music Genre Classification on GTZAN dataset

### **1. Dataset Preparation**

* **Dataset**: GTZAN — 1,000 songs (30s each) across 10 genres. One corrupted file removed → 999 samples.
* **Split**: No predefined validation set → manually split 90% train, 10% test.

In [ ]:
from datasets import load_dataset

gtzan = load_dataset("marsyas/gtzan", "all")
gtzan = gtzan["train"].train_test_split(seed=42, shuffle=True, test_size=0.1)

* **Audio format**: 1D NumPy array, 22,050 Hz sampling rate.
* **Label mapping**: `int2str()` converts numeric genre labels to string labels.

### **2. Pick a Pretrained Model**

* Chosen model: `DistilHuBERT` — smaller, faster distilled version of HuBERT.
* Model type: encoder-only transformer for audio, works with raw waveform input.
* Feature extractor: normalizes audio to **zero mean, unit variance**, and handles attention masks.

In [ ]:
from transformers import AutoFeatureExtractor

model_id = "ntu-spml/distilhubert"
feature_extractor = AutoFeatureExtractor.from_pretrained(
    model_id, do_normalize=True, return_attention_mask=True
)

* **Resample** audio to model’s expected sampling rate (16 kHz for DistilHuBERT).


In [ ]:
from dataset import Audio
sampling_rate = feature_extractor.sampling_rate
gtzan = gtzan.cast_column("audio", Audio(sampling_rate=sampling_rate))


### **3. Preprocessing Audio**

* Truncate longer clips to 30s.
* Normalize and convert to input values + attention masks.


In [ ]:
max_duration = 30.0

def preprocess_function(examples):
    audio_arrays = [x["array"] for x in examples["audio"]]
    inputs = feature_extractor(
        audio_arrays,
        sampling_rate=feature_extractor.sampling_rate,
        max_length=int(feature_extractor.sampling_rate * max_duration),
        truncation=True,
        return_attention_mask=True,
    )
    return inputs

gtzan_encoded = gtzan.map(
    preprocess_function,
    remove_columns=["audio", "file"],
    batched=True,
    batch_size=100,
    num_proc=1,
)
gtzan_encoded = gtzan_encoded.rename_column("genre", "label")

* **Label mapping** for training:

In [ ]:
id2label = {str(i): gtzan_encoded["train"].features["label"].int2str(i) for i in range(10)}
label2id = {v: k for k, v in id2label.items()}

### **4. Model Setup for Classification**
* Use `AutoModelForAudioClassification` to automatically add a classification head.

In [ ]:

from transformers import AutoModelForAudioClassification

model = AutoModelForAudioClassification.from_pretrained(
    model_id,
    num_labels=len(id2label),
    label2id=label2id,
    id2label=id2label,
)

### **5. Training Configuration**

* Define `TrainingArguments` for batch size, learning rate, epochs, etc.
* Enable automatic **Hugging Face Hub** uploads with `push_to_hub=True`.


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    f"{model_id.split('/')[-1]}-finetuned-gtzan",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=1,
    per_device_eval_batch_size=8,
    num_train_epochs=10,
    warmup_ratio=0.1,
    logging_steps=5,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    fp16=True,
    push_to_hub=True,
)

* Metrics: Use **accuracy** since the dataset is balanced.


In [ ]:
import evaluate
import numpy as np

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions = np.argmax(eval_pred.predictions, axis=1)
    return metric.compute(predictions=predictions, references=eval_pred.label_ids)

### **6. Training with the Trainer API**


In [ ]:
from transformers import Trainer

trainer = Trainer(
    model,
    training_args,
    train_dataset=gtzan_encoded["train"],
    eval_dataset=gtzan_encoded["test"],
    tokenizer=feature_extractor,
    compute_metrics=compute_metrics,
)

trainer.train()

* **Tip**: If you encounter CUDA OOM errors, reduce `per_device_train_batch_size` and use `gradient_accumulation_steps` to compensate.
### **7. Upload and Share**

* Push model and training results to Hugging Face Hub:


In [ ]:
kwargs = {
    "dataset_tags": "marsyas/gtzan",
    "dataset": "GTZAN",
    "model_name": f"{model_id.split('/')[-1]}-finetuned-gtzan",
    "finetuned_from": model_id,
    "tasks": "audio-classification",
}

trainer.push_to_hub(**kwargs)


In [ ]:
#* Load the model via pipeline:

from transformers import pipeline

pipe = pipeline("audio-classification", model="your-username/distilhubert-finetuned-gtzan")